# EDA Overview - Final Dataset

This notebook provides a general overview of the final dataset used in the early warning model for CERF conflict allocations. The goal is to understand the structure, coverage, and quality of the data before conducting variable-specific analyses.

The dataset integrates multiple sources (ACLED, IDMC, EconAI, INFORM, HDX Signals, and CERF)  into a single panel with country-month observations. This overview covers:

- Dataset dimensions and structure
- Temporal and geographic coverage
- Missing values by variable and source
- Distribution of the target variable (`allocation-eligible`)

In [4]:
import os
import pandas as pd
import numpy as np

df = pd.read_csv('../data_clean/final_data.csv')
print(f"Data loaded successfully: {df.shape}")

Data loaded successfully: (17424, 100)


In [5]:
# Dataset dimensions
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nColumn types:\n{df.dtypes.value_counts()}")

Rows: 17,424
Columns: 100

Column types:
float64    73
int64      13
bool        8
str         6
Name: count, dtype: int64


In [6]:
# Index structure
print(f"Countries: {df['iso3'].nunique()}")
print(f"Date range: {df['month'].min()} to {df['month'].max()}")
print(f"Unique months: {df['month'].nunique()}")

Countries: 176
Date range: 2018-01-01 to 2026-03-01
Unique months: 99


In [9]:
# See all columns without truncation
pd.set_option('display.max_rows', None)
print(df.columns.tolist())

['iso3', 'month', 'risk_3', 'risk_12', 'logfat_risk_3', 'logfat_risk_12', 'INFORM', 'VU', 'CC', 'HA', 'event_type', 'sub_event_type', 'disorder_type', 'events', 'fatalities', 'hdx_alert_level', 'hdx_value', 'monthly_displacement', 'Flight', 'Airport', 'Travel', 'Train', 'Bus', 'Passport', 'Travel_visa', 'Right_of_asylum', 'rolling_3m_displacements', 'allocation-eligible', 'target_2m', 'risk_gt_06', 'is_protracted', 'disp_6m_avg', 'fat_6m_avg', 'hdx_med_high_count', 'hdx_3m_sum', 'p_sig1', 'p_sig2', 'p_sig3', 'protracted_signal', 'h_sig1', 'h_sig2', 'hard_onset_signal', 'early_signal', 'monthly_displacement_lag1', 'monthly_displacement_lag2', 'fatalities_lag1', 'events_lag1', 'risk_3_lag1', 'risk_12_lag1', 'logfat_risk_3_lag1', 'logfat_risk_12_lag1', 'hdx_alert_max', 'hdx_alert_sum', 'hdx_alert_mean', 'acled_civilian_targeted', 'acled_heavy_warfare', 'acled_social_unrest', 'acled_territorial_shift', 'acled_strategic_or_other', 'acled_political_violence', 'Flight_lag_1m', 'Airport_lag_1m

In [10]:
# Distribution of allocation-eligible
print("=== Allocation-Eligible Distribution ===")
print(df['allocation-eligible'].value_counts())
print(f"\nProportion:")
print((df['allocation-eligible'].value_counts(normalize=True) * 100).round(2))

# Countries most frequently allocation-eligible
print("\n=== Top 20 Countries by Allocation-Eligible Months ===")
print(df[df['allocation-eligible'] == 1].groupby('iso3').size().sort_values(ascending=False).head(20))

=== Allocation-Eligible Distribution ===
allocation-eligible
0    16744
1      680
Name: count, dtype: int64

Proportion:
allocation-eligible
0    96.1
1     3.9
Name: proportion, dtype: float64

=== Top 20 Countries by Allocation-Eligible Months ===
iso3
COD    98
ETH    64
SYR    54
MMR    50
SDN    50
BFA    43
AFG    40
CAF    35
SSD    34
SOM    30
PSE    25
NGA    19
MLI    17
HTI    15
MOZ    14
YEM    12
PHL    11
NER     9
LBN     7
KHM     6
dtype: int64


## Variable Dictionary by Source

### ACLED (Structured)
Variables: `events`, `fatalities`, `event_type`, `sub_event_type`, `disorder_type`,
`acled_civilian_targeted`, `acled_heavy_warfare`, `acled_social_unrest`,
`acled_territorial_shift`, `acled_strategic_or_other`, `acled_political_violence`

Source: `acled_data_wrangled.csv`, aggregated by country-month.
`fatalities` is the monthly sum; `events` is the count of daily records.
The `acled_*` columns are binary indicators for event type categories.

<hr>

### ACLED Notes (NLP / Feature Engineering)
Variables: `Flight`, `Airport`, `Travel`, `Train`, `Bus`, `Passport`,
`Travel_visa`, `Right_of_asylum` + lags, growth rates, and rolling means

Source: Extracted from the `notes` field of ACLED events via keyword-based
text processing in `feature_engineering.ipynb`. These variables capture the
frequency of mobility and displacement-related terms in conflict event reports.

<hr>

### HDX Signals (WFP Market Monitor)
Variables: `hdx_alert_level`, `hdx_value`, `hdx_med_high_count`, `hdx_3m_sum`,
`hdx_alert_max`, `hdx_alert_sum`, `hdx_alert_mean`

Source: `hdx_signals.csv`. Indicator: `wfp_market_monitor`.
Alert levels are "Medium concern" or "High concern". `hdx_value` represents
the percentage increase in the cost of the food basket.

<hr>

### IDMC (Internal Displacement Monitoring Centre)
Variables: `monthly_displacement`, `rolling_3m_displacements`,
`monthly_displacement_lag1`, `monthly_displacement_lag2`

Source: `idmc_conflict_wrangled.csv`, filtered to conflict-induced displacement
only and aggregated from daily to monthly observations.

<hr>

### EconAI
Variables: `risk_3`, `risk_12`, `logfat_risk_3`, `logfat_risk_12`

Source: EconAI conflict forecasting model. `risk_3` and `risk_12` are predicted
probabilities of armed conflict onset at 3 and 12-month horizons.
`logfat_risk_3` and `logfat_risk_12` are predicted fatalities on a log scale.

<hr>

### INFORM Risk Index
Variables: `INFORM`, `VU`, `CC`, `HA`

Source: INFORM Risk Index. `INFORM` is the composite risk score;
`VU` = Vulnerability, `CC` = Coping Capacity, `HA` = Hazard & Exposure.

<hr>

### Target and Constructed Variables
Variables: `allocation-eligible`, `target_2m`, `risk_gt_06`, `is_protracted`,
`disp_6m_avg`, `fat_6m_avg`, `p_sig1`, `p_sig2`, `p_sig3`, `protracted_signal`,
`h_sig1`, `h_sig2`, `hard_onset_signal`, `early_signal`

These variables are constructed during feature engineering and are not
directly sourced from any single external dataset.